In [ ]:
import torch
import matplotlib.pyplot as plt
from pathlib import Path
import sys

# Add src to path (same as training notebooks)
REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT / 'src'))

from dataset import get_data_loaders

# Load exactly as the training notebooks do
train_loader, val_loader, cls2idx, idx2cls = get_data_loaders(
    data_dir=REPO_ROOT / 'data',
    batch_size=32,
    num_workers=0
)

print("=== FUNDAMENTAL DATA CHECKS ===")
print(f"Train dataset size: {len(train_loader.dataset)}")
print(f"Val dataset size: {len(val_loader.dataset)}")
print(f"Expected: 2400 train, 400 val\n")

# Get one batch from train and one from val
train_batch_imgs, train_batch_labels = next(iter(train_loader))
val_batch_imgs, val_batch_labels = next(iter(val_loader))

print(f"Train batch shape: {train_batch_imgs.shape} (should be [32, 3, 224, 224])")
print(f"Val batch shape: {val_batch_imgs.shape}")
print(f"Train labels in batch: {train_batch_labels.unique().tolist()}")
print(f"Val labels in batch: {val_batch_labels.unique().tolist()}\n")

# Check: are train and val tensors actually different?
print("=== CRITICAL: Are train and val images DIFFERENT? ===")
print(f"Train batch mean pixel value (across all channels): {train_batch_imgs.mean():.4f}")
print(f"Val batch mean pixel value (across all channels): {val_batch_imgs.mean():.4f}")
print(f"Are they numerically identical? {torch.allclose(train_batch_imgs, val_batch_imgs)}")
print("(Should be False — if True, data is duplicated)\n")

# Check: is augmentation actually happening?
print("=== CRITICAL: Is augmentation working? ===")
# Get same image twice from train loader (reload iterator)
iter1 = iter(train_loader)
batch1_imgs, batch1_labels = next(iter1)
iter2 = iter(train_loader)
batch2_imgs, batch2_labels = next(iter2)

# Check if two train batches are different (should be due to augmentation + shuffling)
print(f"Are two consecutive train batches identical? {torch.allclose(batch1_imgs, batch2_imgs)}")
print("(Should be False — if True, shuffling/augmentation is broken)\n")

# List actual files on disk to verify structure
print("=== FILES ON DISK ===")
train_dir = REPO_ROOT / 'data' / 'train'
val_dir = REPO_ROOT / 'data' / 'val'

for split, split_dir in [('TRAIN', train_dir), ('VAL', val_dir)]:
    print(f"\n{split}:")
    for cls_dir in sorted(split_dir.iterdir()):
        if cls_dir.is_dir():
            num_files = len(list(cls_dir.glob('*.*')))
            print(f"  {cls_dir.name}: {num_files} files")